# 🔵 [SOLUTION] IBM Quantum Safe Intermediate Lab: Hybrid Cryptography & ML-DSA Signatures

## 🎯 Learning Objectives
1. Understand why **Hybrid Cryptography** is the global gold standard for post-quantum transition.
2. Implement a hybrid key derivation pipeline using **HKDF (Extract & Expand)** to combine classical Elliptic Curve (ECDH) and ML-KEM-768 lattice secrets.
3. Enforce structural validation rules for **ML-DSA-65** digital signatures (FIPS 204).

---
## 🧱 Task 1: Complete the Hybrid Key Derivation Function (HKDF)
In hybrid protocols, a classical secret and a post-quantum secret are generated in parallel. To securely combine them into a single high-entropy symmetric session key, we use **HKDF (Extract & Expand)** based on HMAC-SHA256.

### Mathematical Steps:
1. **Extract**: Combine `classical_ecdh_secret` and `pq_mlkem_secret` by concatenation. Compute a pseudorandom key (PRK) using HMAC-SHA256 with a salt.
   $$\text{PRK} = \text{HMAC-SHA256}(\text{salt}, \text{classical\_secret} \parallel \text{pq\_secret})$$
2. **Expand**: Derive the final shared key (OKM) from the PRK using the application-specific info tag and an ending count block byte `\x01`.
   $$\text{Derived\_Key} = \text{HMAC-SHA256}(\text{PRK}, \text{info} \parallel \text{count})$$
   *Hint*: Use info tag `b"HYBRID-SESSION-KEY-V1"` and count block `b"\x01"` concatenated: `b"HYBRID-SESSION-KEY-V1\x01"`.

In [ ]:
import hmac
import hashlib

def derive_hybrid_session_key(classical_ecdh_secret: bytes, pq_mlkem_secret: bytes, salt: bytes = b"QFF2026-SALT") -> bytes:
    """
    Implements hybrid key derivation by combining classical + post-quantum secrets:
    1. Extract: PRK = HMAC-SHA256(salt, classical_ecdh_secret + pq_mlkem_secret)
    2. Expand: OKM = HMAC-SHA256(PRK, b"HYBRID-SESSION-KEY-V1" + b"\x01")
    Return the 32-byte derived symmetric key.
    """
    if not classical_ecdh_secret or not pq_mlkem_secret:
        raise ValueError("Both secrets must be non-empty bytes")
    
    combined = classical_ecdh_secret + pq_mlkem_secret
    prk = hmac.new(salt, combined, hashlib.sha256).digest()
    derived_key = hmac.new(prk, b"HYBRID-SESSION-KEY-V1\x01", hashlib.sha256).digest()
    return derived_key

---
## 🧱 Task 2: Validate ML-DSA-65 Digital Signatures
In FIPS 204 **ML-DSA-65**, signatures and public keys have fixed, mathematically strict structural sizes. To protect systems against malformed packets or buffer overflow attacks, we must validate bounds before executing expensive algebraic verification:
- **ML-DSA-65 Signatures** must be exactly **3,293 bytes**.
- **ML-DSA-65 Public Keys** must be exactly **1,952 bytes**.
- **Payload** must be non-empty.

In [ ]:
def verify_mldsa_signature(payload: bytes, signature: bytes, public_key: bytes) -> bool:
    """
    Validates an ML-DSA-65 (FIPS 204) signature structure:
    - Signature length must be exactly 3,293 bytes
    - Public key length must be exactly 1,952 bytes
    - Payload must not be empty
    Returns True if valid structure, False otherwise.
    """
    if not payload:
        return False
    if len(signature) != 3293:
        return False
    if len(public_key) != 1952:
        return False
    return True

---
## 🧪 Task 3: Test Your Cryptographic Pipeline
Run the test vectors below to verify your logic.

In [ ]:
# Mock secrets
ecdh_sec = b"classical_ecdh_shared_secret_val"
mlkem_sec = b"pq_lattice_mlkem_shared_secret_val"

# Derive Hybrid Session Key
session_key = derive_hybrid_session_key(ecdh_sec, mlkem_sec)
print(f"Derived 32-byte key: {session_key.hex()}")

# Mock ML-DSA signatures
mock_signature = b"\x00" * 3293
mock_pubkey = b"\x00" * 1952
mock_payload = b"QFF-2026-PAYLOAD-DATA"

is_valid = verify_mldsa_signature(mock_payload, mock_signature, mock_pubkey)
print(f"ML-DSA-65 Signature Structure Validated: {is_valid}")